In [ ]:
import cv2
import numpy as np
import serial
import time

# Initialize camera
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# Initialize serial communication
ser = serial.Serial('/dev/ttyACM0', 115200)
time.sleep(2)

# --- Closed Loop Control Parameters ---
SETPOINT = 10000
MAX_PWM = 255

# Lower Kp gives a smoother ramp up to maximum speed. 
# It's fast, but won't violently jerk the hardware.
Kp = 0.05 

# Deadband prevents jitter when you are very close to the setpoint
DEADBAND = 150 

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture")
            break

        # Convert to grayscale and improve contrast
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.equalizeHist(gray)
        gray = cv2.convertScaleAbs(gray, alpha=1.2, beta=10)

        # Blur and Edge detection
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 50, 150)

        # Count edges
        edge_count = np.sum(edges > 0)

        # Calculate Error
        error = SETPOINT - edge_count

        # Proportional Control Math: Speed is proportional to the error
        raw_speed = abs(error) * Kp

        # Clamp the speed to your max PWM
        pwm_out = int(min(raw_speed, MAX_PWM))

        # Apply Deadband
        if abs(error) < DEADBAND:
            pwm_out = 0

        # Decision Logic & Serial Communication
        if error > 0:  # Edge count is less than 10000 -> FORWARD
            command = f"{pwm_out},0\n"
            ser.write(command.encode('utf-8'))
            print(f"FORWARD  | Edges: {edge_count} | Error: {error} | PWM: {pwm_out}")
            
        elif error < 0:  # Edge count is greater than 10000 -> BACKWARD
            command = f"-{pwm_out},0\n" 
            ser.write(command.encode('utf-8'))
            print(f"BACKWARD | Edges: {edge_count} | Error: {error} | PWM: -{pwm_out}")
            
        else:
            ser.write(b"0,0\n")
            print(f"STOPPED  | Edges: {edge_count}")

except KeyboardInterrupt:
    print("\nManual stop (CTRL+C)")

finally:
    print("Stopping car safely...")
    ser.write(b"0,0\n")  # STOP
    cap.release()
    ser.close()